## Structured Output in LangChain
Normally LLM returns plain text. But sometimes we need the output in a fixed format like JSON or a Pydantic object.

Use `.with_structured_output()` to force the model to follow a specific schema.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [3]:
from langchain_groq import ChatGroq

model = ChatGroq(model='openai/gpt-oss-20b')

### Pydantic schema for structured output
Define a Pydantic `BaseModel` with field types and descriptions. Pass it to `.with_structured_output()`.

Adding `description` to each field is important — it tells the model what value to fill in.

In [4]:
from pydantic import BaseModel, Field

class MovieInfo(BaseModel):
    title: str = Field(description='Movie ka naam')
    director: str = Field(description='Director ka naam')
    year: int = Field(description='Release year, jaise 2020')
    rating: float = Field(description='IMDB rating out of 10')
    genre: str = Field(description='Movie ka genre jaise Action, Drama etc')

# model ko structured output dene ka instruction
structured_model = model.with_structured_output(MovieInfo)

result = structured_model.invoke('Tell me about the movie Inception')
print(result)
print(type(result))

title='Inception' director='Christopher Nolan' year=2010 rating=8.8 genre='Science Fiction, Thriller'
<class '__main__.MovieInfo'>


### Converting result to dictionary
The result is a Pydantic object. Use `.model_dump()` to convert it to a dict.

In [5]:
# Pydantic object ko dict mein convert karna
result_dict = result.model_dump()
print(result_dict)
print()

# individual fields access
print(f'Movie: {result.title}')
print(f'Director: {result.director}')
print(f'Year: {result.year}')
print(f'Rating: {result.rating}')

{'title': 'Inception', 'director': 'Christopher Nolan', 'year': 2010, 'rating': 8.8, 'genre': 'Science Fiction, Thriller'}

Movie: Inception
Director: Christopher Nolan
Year: 2010
Rating: 8.8


### Extracting multiple items — List fields
Use `List` type in a field when you need the model to return multiple values.

In [6]:
from typing import List

class BookRecommendation(BaseModel):
    topic: str = Field(description='Kis topic ke liye books recommend ki hain')
    books: List[str] = Field(description='Recommended books ki list')
    difficulty: str = Field(description='Beginner, Intermediate ya Advanced')

book_model = model.with_structured_output(BookRecommendation)

resp = book_model.invoke('Python seekhne ke liye 3 books recommend karo beginners ke liye')
print(f'Topic: {resp.topic}')
print(f'Difficulty: {resp.difficulty}')
for i, book in enumerate(resp.books, 1):
    print(f'  {i}. {book}')

Topic: Python
Difficulty: Beginner
  1. Automate the Boring Stuff with Python by Al Sweigart
  2. Python Crash Course by Eric Matthes
  3. Learn Python the Hard Way by Zed A. Shaw


### Nested schema
For complex data, use nested Pydantic models. For example, a person object containing an address object.

In [7]:
class Address(BaseModel):
    city: str = Field(description='City ka naam')
    state: str = Field(description='State ka naam')
    country: str = Field(description='Country ka naam')

class PersonInfo(BaseModel):
    name: str = Field(description='Person ka naam')
    age: int = Field(description='Person ki age')
    profession: str = Field(description='Kya kaam karta hai')
    address: Address = Field(description='Person ka address')

person_model = model.with_structured_output(PersonInfo)

resp = person_model.invoke('Sundar Pichai ke baare mein batao')
print(f'Name: {resp.name}')
print(f'Age: {resp.age}')
print(f'Profession: {resp.profession}')
print(f'City: {resp.address.city}')
print(f'Country: {resp.address.country}')

BadRequestError: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '**सुनदर पिचाई** (Sundar Pichai) एक भारतीय-अमेरिकी उद्यमी और प्रौद्योगिकी कार्यकारी हैं, जो **Alphabet Inc.** (गूगल की मूल कंपनी) के वर्तमान **संचालक (CEO)** और **Google के पूर्व CEO** हैं।  \n\n### प्रारम्भिक जीवन\n- **जन्म:** 12 जुलाई 1972, तमिलनाडु, भारत के मयूरपुरम गाँव में।  \n- **शिक्षा:**  \n  - पी.ई. (P.E.) में बी.ई. (B.E.) – सेंट जॉन कॉलेज, चेन्नई।  \n  - एम.एस. (M.S.) – स्टैनफोर्ड विश्वविद्यालय, कंप्यूटर विज्ञान एवं इलेक्ट्रिकल इंजीनियरिंग।  \n  - एम.बी.ए. (MBA) – इम्पीरियल कॉलेज, लंदन (अंतरराष्ट्रीय प्रबंधन कार्यक्रम)।  \n\n### करियर\n| वर्ष | पद | मुख्य योगदान |\n|------|-----|---------------|\n| 2004 | सॉफ्टवेयर इंजीनियर | गूगल में शामिल हुए, प्रोडक्ट मैनेजमेंट में कदम रखा। |\n| 2008 | प्रोडक्ट मैनेजर | गूगल Chrome और Chrome OS के विकास में अहम भूमिका निभाई। |\n| 2014 | Google के 

### Fixed choices using Literal
Use `Literal` to restrict a field to specific allowed values only.

In [ ]:
from typing import Literal

class SentimentResult(BaseModel):
    text: str = Field(description='Original text jo analyze kiya')
    sentiment: Literal['positive', 'negative', 'neutral'] = Field(description='Sentiment of the text')
    confidence: float = Field(description='Confidence score 0 to 1')

sentiment_model = model.with_structured_output(SentimentResult)

texts = [
    'Ye movie bahut zabardast thi! Maza aa gaya',
    'Khaana bilkul bekar tha, dobara nahi jaunga',
    'Theek hai, kuch khaas nahi tha'
]

for t in texts:
    r = sentiment_model.invoke(t)
    print(f'Text: {r.text[:40]}... | Sentiment: {r.sentiment} | Confidence: {r.confidence}')

### Optional fields
Use `Optional` for fields that may or may not be present in the output.

In [8]:
from typing import Optional

class ProductReview(BaseModel):
    product_name: str = Field(description='Product ka naam')
    rating: int = Field(description='Rating 1 se 5 tak')
    pros: List[str] = Field(description='Achhi baatein')
    cons: List[str] = Field(description='Buri baatein')
    suggestion: Optional[str] = Field(default=None, description='Koi suggestion ho toh')

review_model = model.with_structured_output(ProductReview)

resp = review_model.invoke('iPhone 15 ka review do — honest wala')
print(f'Product: {resp.product_name}')
print(f'Rating: {resp.rating}/5')
print(f'Pros: {resp.pros}')
print(f'Cons: {resp.cons}')
if resp.suggestion:
    print(f'Suggestion: {resp.suggestion}')

Product: iPhone 15
Rating: 4/5
Pros: ['Aesthetic design with new color options', 'Improved battery life with faster charging', 'Upgraded camera system with better low-light performance', 'iOS 17 offers new features and smoother performance', 'Build quality remains premium with ceramic shield', 'Enhanced 5G connectivity', 'Face ID remains secure']
Cons: ['Price remains high', 'No significant performance leap over iPhone 14 Pro', 'No new display technology (still OLED 120Hz)', 'Limited storage options', 'No headphone jack or MagSafe improvements', 'No major changes in software ecosystem', 'No significant battery capacity increase', 'Limited availability of accessories', 'No major sustainability improvements', 'No major changes in design']
Suggestion: Consider offering more storage options and a more affordable variant to broaden market reach.


### Structured output with a chain
Combine prompt template with structured model to create a reusable chain.

In [9]:
from langchain_core.prompts import ChatPromptTemplate

class CityInfo(BaseModel):
    name: str = Field(description='City ka naam')
    population: str = Field(description='Approx population')
    famous_for: List[str] = Field(description='Kis cheez ke liye famous hai')
    best_time_to_visit: str = Field(description='Ghumne ka best time')

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Tum ek travel expert ho. Accurate info do.'),
    ('human', '{city} ke baare mein batao')
])

chain = prompt | model.with_structured_output(CityInfo)

result = chain.invoke({'city': 'Jaipur'})
print(f'City: {result.name}')
print(f'Population: {result.population}')
print(f'Famous for: {", ".join(result.famous_for)}')
print(f'Best time: {result.best_time_to_visit}')

BadRequestError: Error code: 400 - {'error': {'message': "Tool call validation failed: tool call validation failed: parameters for tool CityInfo did not match schema: errors: [missing properties: 'population', 'famous_for', 'best_time_to_visit']", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "CityInfo", "arguments": {"name":"Jaipur"}}'}}

In [ ]:
# same chain, different city
result2 = chain.invoke({'city': 'Varanasi'})
print(f'City: {result2.name}')
print(f'Population: {result2.population}')
print(f'Famous for: {", ".join(result2.famous_for)}')
print(f'Best time: {result2.best_time_to_visit}')

### TypedDict alternative
If you don't want to use Pydantic, `TypedDict` also works. But Pydantic is better because it has built-in validation.

In [10]:
from typing import TypedDict, Annotated

class JokeOutput(TypedDict):
    setup: Annotated[str, 'joke ka setup part']
    punchline: Annotated[str, 'joke ka punchline']
    category: Annotated[str, 'joke ka category jaise tech, dad joke etc']

joke_model = model.with_structured_output(JokeOutput)

joke = joke_model.invoke('ek programming wala joke sunao')
print(f'Setup: {joke["setup"]}')
print(f'Punchline: {joke["punchline"]}')
print(f'Category: {joke["category"]}')

Setup: Why do programmers prefer dark mode?
Punchline: Because light attracts bugs.
Category: programming


### Error handling
Sometimes the model doesn't follow the schema properly. Use try-except to handle these cases.

In [11]:
# error handling example
try:
    result = structured_model.invoke('Tell me about the movie 3 Idiots')
    print(f'Movie: {result.title}')
    print(f'Director: {result.director}')
    print(f'Year: {result.year}')
    print(f'Rating: {result.rating}')
    print(f'Genre: {result.genre}')
except Exception as e:
    print(f'Error aaya: {e}')
    print('Retry karo ya model change karo')

Movie: 3 Idiots
Director: Rajkumar Hirani
Year: 2009
Rating: 8.4
Genre: Comedy, Drama


In [12]:
#data classes

# data class is a class typically containing mainly data although there aren't really any 
# restrictions , we can create it using the @data class decorator

In [13]:
from pydantic import BaseModel ,Field
from langchain.agents import create_agent
class contact_info(BaseModel):
    """ contact information """
    name: str= Field(description='name of person')
    email:str= Field(description=' email of the person')
    phone:str= Field(description='give your mobile numver')

In [21]:
from langchain.chat_models import init_chat_model

llm = init_chat_model('openai/gpt-oss-20b', model_provider='groq')
model = create_agent(model=llm, response_format=contact_info)


In [23]:
model.invoke({
    'messages':[{'role':'user','content':'extract information about mohiht payasi'}]
})

{'messages': [HumanMessage(content='extract information about mohiht payasi', additional_kwargs={}, response_metadata={}, id='c9763089-27d2-4763-ba67-2a1c99da731f'),
  AIMessage(content='{"name":"","email":"","phone":""}', additional_kwargs={'reasoning_content': 'The user wants to extract information about "mohih payasi" (maybe "Mohih Payasi"?). The schema is "contact_info" which requires name, email, phone. But the user didn\'t provide any data. We need to output JSON that matches the schema, with all required fields. We don\'t have any data. So we need to produce placeholders? The instructions: "Before you respond, verify the JSON is valid: ... Then output ONLY the final JSON object." We must output JSON that matches the schema. Since we don\'t have actual data, we could put placeholders like empty strings? But the required fields cannot be empty? They can be empty strings, but must be present. So we can produce:\n\n{\n  "name": "",\n  "email": "",\n  "phone": ""\n}\n\nBut is that va

In [24]:
model.invoke({
    'messages':[{'role':'user','content':'extract information about mohiht payasi'}]
})['structured_response']

contact_info(name='Mohiht Payasi', email='mohih.payasi@example.com', phone='+1234567890')

In [25]:
!pip install typedict

  Using cached typedict-0.1.0-py3-none-any.whl.metadata (8.2 kB)
Using cached typedict-0.1.0-py3-none-any.whl (6.6 kB)


In [28]:
from typedict import TypeDict
from langchain.agents import create_agent
class contact_info_1(TypeDict):
    """ contact information """
    name: str= 'name of person'
    email:str= ' email of the person'
    phone:str= 'give your mobile numver'

llm = init_chat_model('openai/gpt-oss-20b', model_provider='groq')
model = create_agent(model=llm, response_format=contact_info_1)

ValueError: Unsupported schema type: <class 'abc.ABCMeta'>. Supported types: Pydantic models, dataclasses, TypedDicts, and JSON schema dicts.